In [ ]:
# Example 1: verify the primary public API (configure_azure_monitor) is importable and callable.
from azure.monitor.opentelemetry import configure_azure_monitor
import inspect

assert callable(configure_azure_monitor), "configure_azure_monitor must be callable"

sig = inspect.signature(configure_azure_monitor)
print(f"configure_azure_monitor signature: {sig}")
print(f"Parameters: {list(sig.parameters.keys())}")

# The function accepts **kwargs for various configuration options
assert sig.parameters, "configure_azure_monitor should accept parameters"


In [ ]:
# Example 2: verify the underlying exporter dependency is reachable.
from azure.monitor.opentelemetry.exporter import (
    AzureMonitorLogExporter,
    AzureMonitorMetricExporter,
    AzureMonitorTraceExporter,
)

for cls in (AzureMonitorLogExporter, AzureMonitorMetricExporter, AzureMonitorTraceExporter):
    assert isinstance(cls, type), f"{cls!r} should be a class"
    print(f"{cls.__name__} -> {cls.__module__}")


In [ ]:
# Example 3: verify OpenTelemetry SDK trace/metric/log primitives wire together via this distro.
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

provider = TracerProvider()
exporter = InMemorySpanExporter()
provider.add_span_processor(SimpleSpanProcessor(exporter))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("azure-monitor-test")
with tracer.start_as_current_span("smoke-span") as span:
    span.set_attribute("smoke.test", True)

spans = exporter.get_finished_spans()
assert len(spans) == 1, f"expected 1 span, got {len(spans)}"
assert spans[0].name == "smoke-span"
print(f"Exported span: {spans[0].name}, attrs={dict(spans[0].attributes)}")


In [ ]:
# Example 4: verify the Azure resource detector is importable and runnable.
from opentelemetry.resource.detector.azure.app_service import AzureAppServiceResourceDetector
from opentelemetry.sdk.resources import Resource

# The distro bundles Azure resource detectors (via the exporter). The App Service
# detector is the canonical one; it runs cleanly even off-Azure, returning an
# empty Resource when the App Service env vars are absent.
detector = AzureAppServiceResourceDetector()
detected = detector.detect()
assert isinstance(detected, Resource)
print(f"AzureAppServiceResourceDetector.detect() -> {dict(detected.attributes)}")

# A basic OTel Resource can be created + merged with a detected one (the distro
# relies on this Resource-merge path during configure_azure_monitor()).
resource = Resource.create({"service.name": "smoke-test", "service.version": "0.0.1"})
merged = resource.merge(detected)
assert merged.attributes["service.name"] == "smoke-test"
print(f"merged resource attributes: {dict(merged.attributes)}")

In [ ]:
# Example 5: verify the distro's bundled instrumentation entry-points are importable.
import importlib

instrumentation_modules = [
    "opentelemetry.instrumentation.requests",
    "opentelemetry.instrumentation.urllib",
    "opentelemetry.instrumentation.urllib3",
    "opentelemetry.instrumentation.logging",
]

loaded = []
for mod_name in instrumentation_modules:
    mod = importlib.import_module(mod_name)
    loaded.append((mod_name, mod.__name__))
    print(f"OK: {mod_name}")

assert len(loaded) == len(instrumentation_modules), "all bundled instrumentations must import"
print(f"\nLoaded {len(loaded)} bundled instrumentation modules cleanly")
